# Fehlerbehandlung: Exceptions in Python

Bisher haben wir uns meist mit Code beschäftigt, der wie gewünscht funktioniert. In der Praxis läuft aber nicht immer alles nach Plan: Eine Datei fehlt, eine Nutzereingabe ist ungültig, eine Division durch Null wird versucht. Python signalisiert solche Probleme zur Laufzeit über sogenannte **Exceptions** (Ausnahmen), vergleichbar mit einem Alarm, der auslöst, wenn während der Ausführung etwas nicht wie erwartet funktioniert. Ignorierst du den Alarm, stürzt dein Programm ab. Du kannst aber auch gezielt reagieren. Das nennen wir **Fehlerbehandlung**.

Du kennst `try`/`except` bereits in Ansätzen aus der Session "Kontrollstrukturen". In dieser Session schauen wir uns Exceptions im Detail an: was sie eigentlich sind, wie man sie gezielt abfängt, welche am häufigsten auftreten und vor allem, wann man einen Fehler abfangen, wann man ihn bewusst crashen lassen und wann man selbst einen auslösen sollte.

⚠️ Viele Beispielzellen in diesem Notebook brechen **absichtlich** mit einer Exception ab, um den jeweiligen Fehler zu zeigen. "Run All" stoppt deshalb an der ersten dieser Zellen. Die Zellen lassen sich stattdessen einzeln nacheinander ausführen.

## Lernziele und Inhalt

**Lernziele**

1. Syntaxfehler und Exceptions unterscheiden und einen Traceback lesen.
2. Exceptions mit `try`/`except` gezielt abfangen und über `as` auf das Exception-Objekt zugreifen.
3. Mehrere Exception-Typen getrennt oder gemeinsam behandeln.
4. Häufige eingebaute Exception-Typen erkennen und ihren typischen Ursachen zuordnen.
5. Mit `raise` eigene Exceptions auslösen und abgefangene Exceptions erneut auslösen.
6. Entscheiden, ob eine Exception abgefangen, durchgelassen oder selbst ausgelöst werden sollte.
7. (Optional) `finally`, `else`, `assert` und eigene Exception-Klassen einsetzen.

**Inhalt**

- [1. Was sind Exceptions?](#was-sind-exceptions)
- [2. Exceptions abfangen mit `try`/`except`](#try-except)
  - [2.1 Das Exception-Objekt und `as`](#exception-objekt)
  - [2.2 Mehrere Exception-Typen behandeln](#mehrere-typen)
- [3. Häufige Exception-Typen](#haeufige-typen)
- [4. Eigene Exceptions auslösen: `raise`](#raise)
  - [4.1 Erneut auslösen (Re-raise)](#re-raise)
- [5. Faustregeln: Abfangen, crashen lassen oder selbst raisen?](#faustregeln)
- [6. Zusammenfassung](#zusammenfassung)
- [7. Vertiefung: `finally`, `else` und `assert` (Optional)](#vertiefung)
  - [Eigene Exception-Klassen](#eigene-klassen)
- [Übungsaufgaben](#uebungsaufgaben)

<a id="was-sind-exceptions"></a>

## 1. Was sind Exceptions?

Python unterscheidet grob zwei Arten von Fehlern:

- **Syntaxfehler** (`SyntaxError`): Der Code verstößt gegen die Grammatik der Sprache und kann gar nicht erst ausgeführt werden. Das ist vergleichbar mit einem Grammatikfehler in einem Satz, zum Beispiel ein fehlender Doppelpunkt.
- **Exceptions** (Ausnahmen): Der Code ist syntaktisch korrekt, aber **während der Ausführung** tritt ein Problem auf, das den normalen Ablauf unterbricht.

Beim Syntaxfehler bemängelt Python den fehlenden Doppelpunkt, bevor auch nur eine Zeile ausgeführt wird:

In [ ]:
if True
    print("Hallo")

Hier ist die Syntax dagegen korrekt. Erst bei der Ausführung stellt Python fest, dass es keinen Namen `Tru` gibt, und löst eine Exception (`NameError`) aus:

In [ ]:
if Tru:
    print("Hallo")

Wird eine Exception nicht behandelt, bricht das Programm an genau dieser Stelle ab. Im folgenden Beispiel wird `result_three` deshalb nie berechnet:

In [ ]:
result_one = 1 + 2
result_two = 10 / 0
result_three = 3 + 4

Diese Fehlermeldung nennt man einen **Traceback**. Er verrät dir mehrere wichtige Dinge:

- **Wo** der Fehler aufgetreten ist (Datei und Zeile, bei mehreren verschachtelten Funktionsaufrufen sogar der komplette Aufrufpfad, von unten nach oben gelesen).
- **Welche Art** von Fehler aufgetreten ist (hier: `ZeroDivisionError`).
- **Welche Nachricht** dazu mitgegeben wurde (hier: "division by zero").

💡 Eine Exception ist, wie fast alles in Python, ein **Objekt** — mit einem Typ (z.B. `ZeroDivisionError`) und meist einer erklärenden Nachricht.

Ruft eine Funktion eine andere auf, wird der Traceback länger: Er zeigt dir den kompletten Aufrufpfad, von unten (wo der Fehler tatsächlich passiert ist) nach oben (bis zum ursprünglichen Aufruf).

In [ ]:
def calculate_ratio(dividend, divisor):
    return dividend / divisor

def calculate_total_ratio(values, divisor):
    total = sum(values)
    return calculate_ratio(total, divisor)

calculate_total_ratio([10, 20, 30], 0)

<a id="try-except"></a>

## 2. Exceptions abfangen mit `try`/`except`

Mit `try`/`except` kannst du auf eine Exception reagieren, statt das Programm abstürzen zu lassen. Der Code, der möglicherweise fehlschlägt, steht im `try`-Block. Tritt dort eine Exception des angegebenen Typs auf, wird die Ausführung des `try`-Blocks sofort abgebrochen und mit dem passenden `except`-Block fortgesetzt.

In [ ]:
try:
    result = 10 / 0
    print("Diese Zeile wird nicht mehr erreicht.")
except ZeroDivisionError:
    print("Division durch Null ist nicht erlaubt.")

print("Das Programm läuft normal weiter.")

⚠️ Gibst du hinter `except` **keinen** Exception-Typ an (`except:`) oder den sehr allgemeinen Typ `Exception` (`except Exception:`), fängst du (fast) **jede** Exception ab — auch solche, die du eigentlich nicht erwartet oder gar nicht behandeln kannst. Das verschleiert Fehler, statt sie sichtbar zu machen. Gib deshalb (fast) immer einen konkreten Exception-Typ an.

<a id="exception-objekt"></a>

### 2.1 Das Exception-Objekt und `as`

Mit `except ExceptionTyp as e` bekommst du das Exception-Objekt selbst in eine Variable (üblicherweise `e` genannt) und kannst z.B. seine Nachricht auslesen:

In [ ]:
try:
    result = 10 / 0
except ZeroDivisionError as e:
    print(f"Ein Fehler ist aufgetreten: {e}")
    print(f"Exception-Typ: {type(e).__name__}")

Neben `type(e).__name__` bringt jedes Exception-Objekt noch `.args` mit, ein Tupel mit den Werten, die beim Auslösen übergeben wurden (meist nur die Nachricht). Manche Exception-Typen ergänzen zusätzliche, speziellere Attribute: `FileNotFoundError` z.B. kennt `.filename`, den Pfad der fehlenden Datei. Im Beispiel versucht `open()`, eine Datei zu öffnen; wie das im Detail funktioniert, zeigt die nächste Session "Dateien lesen und schreiben".

In [ ]:
try:
    open("does_not_exist.txt")
except FileNotFoundError as e:
    print(f"e.args: {e.args}")
    print(f"e.filename: {e.filename}")

<a id="mehrere-typen"></a>

### 2.2 Mehrere Exception-Typen behandeln

Ein `try`-Block kann mehrere unterschiedliche Exceptions auslösen, auf die du ggf. unterschiedlich reagieren willst. Dafür kannst du mehrere `except`-Blöcke hintereinander angeben:

In [ ]:
def safe_divide(dividend, divisor):
    try:
        return dividend / divisor
    except ZeroDivisionError:
        print("Division durch Null ist nicht erlaubt.")
    except TypeError:
        print("Beide Werte müssen Zahlen sein.")

safe_divide(10, 0)
safe_divide(10, "zwei")
print(safe_divide(10, 2))

Sollen mehrere Exception-Typen aber **gleich** behandelt werden, kannst du sie stattdessen als Tupel in einem einzigen `except` zusammenfassen:

In [ ]:
def safe_divide(dividend, divisor):
    try:
        return dividend / divisor
    except (ZeroDivisionError, TypeError) as e:
        print(f"Division nicht möglich: {e}")

safe_divide(10, 0)
safe_divide(10, "zwei")

<a id="haeufige-typen"></a>

## 3. Häufige Exception-Typen

Python bringt viele eingebaute Exception-Typen mit. Die häufigsten, die dir in der Praxis begegnen werden:

- **`ValueError`**: Ein Wert hat den richtigen Typ, aber einen ungültigen Inhalt (z.B. `int("abc")`).
- **`TypeError`**: Eine Operation wird auf einen Typ angewendet, der dafür nicht geeignet ist (z.B. `"5" + 5`).
- **`IndexError`**: Zugriff auf einen Listen-/Tupel-Index, der nicht existiert.
- **`KeyError`**: Zugriff auf einen Dictionary-Schlüssel, der nicht existiert.
- **`ZeroDivisionError`**: Division (oder Modulo) durch `0`.
- **`AttributeError`**: Zugriff auf ein Attribut oder eine Methode, die ein Objekt nicht besitzt.
- **`NameError`**: Verwendung eines Variablen- oder Funktionsnamens, der nicht definiert ist.
- **`FileNotFoundError`**: Zugriff auf eine Datei, die nicht existiert (Thema der nächsten Session "Dateien lesen und schreiben").

Schauen wir uns ein paar davon direkt an:

In [ ]:
int("abc")

In [ ]:
numbers = [1, 2, 3]
numbers[10]

In [ ]:
person = {"name": "Anna"}
person["age"]

💡 `TypeError`, `AttributeError` und `NameError` funktionieren nach demselben Prinzip — probier sie bei Bedarf selbst aus, z.B. mit `"5" + 5`, `"hallo".nicht_vorhanden()` oder dem Aufruf einer nie definierten Variable.

Exceptions kommen nicht nur aus deinem eigenen Code — auch Standardmodule wie `statistics` lösen sie aus. Der Traceback zeigt dann sogar eine Zeile aus dem Quellcode des Moduls selbst:

In [ ]:
import statistics

statistics.mean([])

Das Gleiche gilt für Bibliotheken wie `numpy` oder `pandas`, die dir in den DSC-Kursen häufig begegnen — auch sie lösen dir bekannte Exception-Typen aus, sobald etwas nicht passt:

In [ ]:
import numpy as np

array = np.array([1, 2, 3])
array.reshape(2, 2)

In [ ]:
import pandas as pd

pd.read_csv("nicht_vorhanden.csv")

<a id="raise"></a>

## 4. Eigene Exceptions auslösen: `raise`

Nicht nur Python selbst löst Exceptions aus — auch du kannst das gezielt tun, wenn deine eigene Funktion erkennt, dass etwas nicht stimmt. Das ist besonders nützlich, um **ungültige Eingaben** früh und mit einer klaren Fehlermeldung zurückzuweisen, statt später mit falschen Werten weiterzurechnen.

In [ ]:
def calculate_average(numbers):
    """Return the average of a list of numbers."""
    if not numbers:
        raise ValueError("Die Liste darf nicht leer sein.")
    return sum(numbers) / len(numbers)

print(calculate_average([10, 20, 30]))

In [ ]:
calculate_average([])


Die aufrufende Stelle kann diese Exception nun ganz normal mit `try`/`except` behandeln:

In [ ]:
try:
    average = calculate_average([])
except ValueError as e:
    print(f"Ungültige Eingabe: {e}")
    average = 0

print(f"{average = }")

<a id="re-raise"></a>

### 4.1 Erneut auslösen (Re-raise)

Manchmal willst du auf eine Exception reagieren (z.B. sie protokollieren), sie aber trotzdem weiterhin nach oben durchreichen, damit sie dort behandelt werden kann. Dafür genügt ein `raise` ohne Argument innerhalb des `except`-Blocks:

In [ ]:
def calculate_average_logged(numbers):
    try:
        return calculate_average(numbers)
    except ValueError:
        print("Achtung: calculate_average() wurde mit ungültigen Daten aufgerufen.")
        raise  # Löst dieselbe Exception erneut aus

calculate_average_logged([])

<a id="faustregeln"></a>

## 5. Faustregeln: Abfangen, crashen lassen oder selbst raisen?

Nicht jede Exception sollte abgefangen werden, und nicht jeder Fehlerfall sollte einfach durchlaufen. Ein paar Faustregeln, die dir bei der Entscheidung helfen:

- **Abfangen**, wenn du eine sinnvolle, konkrete Reaktion auf **genau diesen** Fehlerfall hast — z.B. einen Standardwert verwenden, es erneut versuchen oder eine verständliche Meldung statt eines Tracebacks ausgeben. Beispiel: `FileNotFoundError` beim Einlesen einer optionalen Konfigurationsdatei.
- **Crashen lassen** (also *nicht* abfangen), wenn die Exception auf einen **Programmierfehler** hinweist, den du eigentlich beheben solltest, statt ihn zu verschleiern. Ein `TypeError`, weil eine Funktion falsch aufgerufen wurde, soll dir laut auffallen, nicht in einem `except Exception: pass` verschwinden.
- **Selbst raisen**, wenn *deine* Funktion erkennt, dass eine Vorbedingung verletzt ist, und die aufrufende Stelle darüber informiert werden muss, z.B. bei einer ungültigen Eingabe wie in Kapitel 4.

⚠️ Das größte Antipattern der Fehlerbehandlung: Exceptions "verschlucken", indem man sie breit abfängt und nichts (oder nur `pass`) damit tut. So werden Fehler unsichtbar und tauchen später an ganz anderer Stelle als verwirrendes Folgeproblem wieder auf.

💡 Merkregel: Fange nur Exceptions ab, auf die du **an dieser Stelle im Code** sinnvoll reagieren kannst. Alles andere lässt du bewusst weiterlaufen (propagieren), notfalls bis ganz nach oben, wo es zu einem sichtbaren Programmabsturz mit Traceback kommt. Das ist kein Bug, sondern gewolltes Verhalten: Ein sichtbarer Crash ist fast immer besser als ein Programm, das mit falschen Daten unbemerkt weiterläuft.

<a id="zusammenfassung"></a>

## 6. Zusammenfassung

Exceptions sind Fehler, die erst während der Ausführung auftreten. Bleiben sie unbehandelt, brechen sie das Programm mit einem Traceback ab. Mit `try`/`except` lässt sich gezielt auf erwartete Fehlerfälle reagieren, mit `raise` meldet eine eigene Funktion ungültige Eingaben an die aufrufende Stelle.

### Behandelte Konzepte

1. Syntaxfehler (Code wird gar nicht ausgeführt) von Exceptions (Fehler zur Laufzeit) unterscheiden.
2. Einen Traceback von unten nach oben lesen: Exception-Typ, Nachricht und Aufrufpfad.
3. Exceptions mit `try`/`except` abfangen und mit `except Typ as e` auf das Exception-Objekt zugreifen.
4. Mehrere Exception-Typen über mehrere `except`-Blöcke unterschiedlich oder über ein Tupel gemeinsam behandeln.
5. Häufige Exception-Typen erkennen: `ValueError`, `TypeError`, `IndexError`, `KeyError`, `ZeroDivisionError`, `AttributeError`, `NameError`, `FileNotFoundError`.
6. Mit `raise` ungültige Eingaben in eigenen Funktionen zurückweisen und mit einem bloßen `raise` im `except`-Block die aktuelle Exception erneut auslösen.
7. Exceptions nur dort abfangen, wo sinnvoll reagiert werden kann, und sie sonst bewusst durchlaufen lassen, statt sie zu verschlucken.

### Kurzreferenz

| Schreibweise | Bedeutung |
|---|---|
| `try:` … `except ValueError:` | `ValueError` im `try`-Block abfangen |
| `except ValueError as e:` | Exception-Objekt in der Variable `e` verfügbar machen |
| mehrere `except`-Blöcke | verschiedene Exception-Typen unterschiedlich behandeln |
| `except (ValueError, TypeError):` | mehrere Exception-Typen gleich behandeln |
| `raise ValueError("Nachricht")` | selbst eine Exception auslösen |
| `raise` (im `except`-Block) | die abgefangene Exception erneut auslösen |
| `finally:` | Block wird immer ausgeführt (Vertiefung) |
| `else:` (nach `except`) | Block wird nur ohne Exception ausgeführt (Vertiefung) |
| `assert condition, "Nachricht"` | interne Annahme prüfen, löst sonst `AssertionError` aus (Vertiefung) |
| `class MyError(Exception):` | eigenen Exception-Typ definieren (Vertiefung) |

<a id="vertiefung"></a>

## 7. Vertiefung: `finally`, `else` und `assert` (Optional)

### `finally`

Ein `finally`-Block wird **immer** ausgeführt, egal, ob im `try`-Block eine Exception aufgetreten ist oder nicht. Das eignet sich für Aufräumarbeiten, die auf jeden Fall passieren müssen (z.B. eine Ressource freigeben):

In [ ]:
def process(value):
    try:
        print(f"Verarbeite {value}...")
        result = 100 / value
        print(f"Ergebnis: {result}")
    except ZeroDivisionError:
        print("Division durch Null übersprungen.")
    finally:
        print("Verarbeitung beendet.\n")

process(10)
process(0)

### `else` bei `try`

Ein `else`-Block nach `try`/`except` wird nur ausgeführt, wenn **keine** Exception aufgetreten ist — ähnlich wie du es schon von `while`-Schleifen kennst:

In [ ]:
try:
    result = 10 / 2
except ZeroDivisionError:
    print("Division durch Null ist nicht erlaubt.")
else:
    print(f"Berechnung erfolgreich: {result}")

### `assert`

Mit `assert` kannst du eine Bedingung prüfen, die eigentlich **immer** wahr sein sollte — ist sie es nicht, deutet das auf einen Bug in deinem eigenen Code hin, nicht auf eine ungültige Nutzereingabe. Ist die Bedingung `False`, löst `assert` eine `AssertionError`-Exception aus:

Im folgenden Beispiel prüft `raise` die Eingabe, `assert` dagegen eine Annahme über das eigene Rechenergebnis. Die Funktion enthält absichtlich einen Bug, den `assert` aufdeckt:

In [ ]:
def calculate_discounted_price(price, discount_percent):
    """Return the price after subtracting discount_percent percent."""
    if discount_percent < 0 or discount_percent > 100:
        raise ValueError("Der Rabatt muss zwischen 0 und 100 Prozent liegen.")

    discounted_price = price * (1 + discount_percent / 100)  # Bug: + statt -

    assert discounted_price <= price, "Der rabattierte Preis darf nie über dem Originalpreis liegen."
    return discounted_price

calculate_discounted_price(100, 20)

⚠️ Verwende `assert` nicht, um Nutzereingaben zu validieren — dafür ist `raise` (siehe Kapitel 4) das richtige Werkzeug. `assert`-Anweisungen können beim optimierten Ausführen von Python (`python -O`) sogar komplett deaktiviert werden und sollten deshalb nur interne Annahmen ("das darf laut meiner eigenen Logik nie passieren") prüfen, nicht Fehlerfälle, mit denen du rechnen musst.

<a id="eigene-klassen"></a>

### Eigene Exception-Klassen

Bisher hast du beim `raise` immer eingebaute Exception-Typen wie `ValueError` verwendet. Mit dem `class`-Schlüsselwort kannst du aber auch eigene Exception-Typen definieren — dafür erbst du einfach von `Exception`:

In [ ]:
class InsufficientFundsError(Exception):
    """Raised when an account balance is too low for a withdrawal."""

def withdraw(balance, amount):
    if amount > balance:
        raise InsufficientFundsError(f"Nicht genug Guthaben: {balance}€ vorhanden, {amount}€ angefordert.")
    return balance - amount

withdraw(100, 150)

In [ ]:
class InsufficientFundsError(Exception):
    """Raised when an account balance is too low for a withdrawal."""

    def __init__(self, balance, amount):
        self.balance = balance
        self.amount = amount
        super().__init__(f"Nicht genug Guthaben: {balance}€ vorhanden, {amount}€ angefordert.")

def withdraw(balance, amount):
    if amount > balance:
        raise InsufficientFundsError(balance, amount)
    return balance - amount

try:
    withdraw(100, 150)
except InsufficientFundsError as e:
    print(f"Abbruch: {e}")
    print(f"Fehlbetrag: {e.amount - e.balance}€")

💡 Für einfache Fälle reicht meist schon eine Exception-Klasse ohne eigene `__init__`-Methode, wie im ersten Beispiel, die Nachricht allein genügt oft. Ihre Stärken spielen eigene Exceptions vor allem in größeren Projekten aus: über eigene Hierarchien (z.B. mehrere spezifische Fehlerklassen, die von einer gemeinsamen `AppError`-Basisklasse erben) und die Verkettung mit `raise ... from ...`. Diese Themen behandelt die Python Advanced Session "Fehlerbehandlung mit Exceptions".

<a id="uebungsaufgaben"></a>

## Übungsaufgaben

Die folgenden Aufgaben greifen alle Kapitel des Notebooks noch einmal auf. Unter jeder Aufgabe steht eine Codezelle mit einem Anfang, der vervollständigt werden soll. Die Kommentare `# erwartet: ...` zeigen, welches Ergebnis herauskommen sollte.

Die Aufgaben sind unabhängig von den Beispielzellen oben und lassen sich direkt ausführen. Zellen, die vor dem Lösen absichtlich mit einer Exception abbrechen, sind in der Aufgabenstellung markiert.

### Exceptions und Tracebacks

#### Aufgabe 1: Exception-Typ vorhersagen

Jede der auskommentierten Zeilen löst eine andere Exception aus. Überlege **vor** dem Ausführen, welcher Exception-Typ jeweils entsteht. Entferne danach die Kommentarzeichen Zeile für Zeile, führe die Zelle aus und vergleiche mit deiner Vermutung.

In [ ]:
prices = {"Kaffee": 4.99, "Brot": 2.49}
numbers = [10, 20, 30]

# TODO: Vermutung notieren, dann jeweils eine Zeile auskommentieren und prüfen
# int("3.5")             # Vermutung:
# "Preis: " + 9.99       # Vermutung:
# numbers[3]             # Vermutung:
# prices["Käse"]         # Vermutung:
# "hallo".append("!")    # Vermutung:
# print(undefined_total) # Vermutung:
# 10 % 0                 # Vermutung:

#### Aufgabe 2: Traceback lesen

⚠️ Die folgende Zelle bricht **absichtlich** mit einer Exception ab.

Führe die Zelle aus und beantworte anhand des Tracebacks:

1. Welcher Exception-Typ ist aufgetreten und welche Nachricht wird angezeigt?
2. In welcher Funktion und in welcher Zeile ist der Fehler tatsächlich passiert?
3. Über welche Funktionsaufrufe ist das Programm dorthin gelangt?

Behebe den Fehler anschließend, indem du für das fehlende Produkt einen Preis von `1.99` ergänzt.

In [ ]:
def get_unit_price(prices, product):
    return prices[product]


def calculate_line_total(prices, product, quantity):
    unit_price = get_unit_price(prices, product)
    return unit_price * quantity


def calculate_order_total(prices, order):
    total = 0
    for product, quantity in order.items():
        total += calculate_line_total(prices, product, quantity)
    return total


prices = {"Kaffee": 5.0, "Brot": 2.5, "Käse": 7.0}
order = {"Kaffee": 2, "Brot": 1, "Butter": 1}

print(calculate_order_total(prices, order))   # erwartet nach der Korrektur: 14.49

### Exceptions abfangen

#### Aufgabe 3: Text sicher in eine Zahl umwandeln

Schreibe eine Funktion `to_number(text)`, die `text` mit `float()` in eine Zahl umwandelt und zurückgibt. Ist das nicht möglich, soll die Funktion den `ValueError` abfangen und `None` zurückgeben.

In [ ]:
def to_number(text):
    """Return text as float or None if text is not a valid number."""
    # TODO: Umwandlung in einem try-Block versuchen, ValueError abfangen
    pass


print(to_number("3.5"))     # erwartet: 3.5
print(to_number("42"))      # erwartet: 42.0
print(to_number("zwölf"))   # erwartet: None

#### Aufgabe 4: Ungültige Einträge überspringen

Die Liste `raw_ages` stammt aus einem Formular und enthält einige ungültige Einträge. Wandle jeden Eintrag mit `int()` um und hänge ihn an `valid_ages` an. Ungültige Einträge sollen übersprungen werden. Gib für jeden übersprungenen Eintrag den Wert und die Nachricht des Exception-Objekts aus.

Erwartete Ausgabe:
```
Übersprungen: 'acht' (invalid literal for int() with base 10: 'acht')
Übersprungen: '' (invalid literal for int() with base 10: '')
[34, 27, 51]
```

In [ ]:
raw_ages = ["34", "acht", "27", "", "51"]
valid_ages = []

for raw_age in raw_ages:
    # TODO: int() in try versuchen, bei ValueError mit "as e" eine Meldung ausgeben
    pass

print(valid_ages)

#### Aufgabe 5: Unterschiedlich auf Fehler reagieren

`scores` speichert für jede Person die Punkte aus mehreren Runden. Schreibe eine Funktion `get_score(scores, name, round_index)`, die die Punkte einer Person in einer Runde zurückgibt. Fange zwei Fehlerfälle in **getrennten** `except`-Blöcken ab und gib jeweils `None` zurück:

- Die Person existiert nicht: Ausgabe `Unbekannte Person: <name>`
- Die Runde existiert nicht: Ausgabe `Für <name> gibt es keine Runde mit Index <round_index>.`

In [ ]:
scores = {"Anna": [8, 9, 7], "Ben": [6, 10]}


def get_score(scores, name, round_index):
    """Return the score of name in the given round or None if it does not exist."""
    # TODO: Zugriff in try versuchen, KeyError und IndexError getrennt behandeln
    pass


print(get_score(scores, "Anna", 1))   # erwartet: 9
print(get_score(scores, "Tom", 0))    # erwartet: Unbekannte Person: Tom, danach None
print(get_score(scores, "Ben", 2))    # erwartet: Für Ben gibt es keine Runde mit Index 2., danach None

#### Aufgabe 6: Mehrere Fehler gleich behandeln

Schreibe eine Funktion `calculate_unit_price(price_text, quantity_text)`, die beide Texte mit `float()` umwandelt und den Preis pro Stück zurückgibt. Sowohl ein `ValueError` (ungültiger Text) als auch ein `ZeroDivisionError` (Menge `0`) sollen in **einem** `except`-Block abgefangen werden. In beiden Fällen soll `Berechnung nicht möglich: <Nachricht>` ausgegeben und `None` zurückgegeben werden.

In [ ]:
def calculate_unit_price(price_text, quantity_text):
    """Return the price per unit or None if the calculation is not possible."""
    # TODO
    pass


print(calculate_unit_price("10", "4"))      # erwartet: 2.5
print(calculate_unit_price("10", "0"))      # erwartet: Berechnung nicht möglich: float division by zero, danach None
print(calculate_unit_price("zehn", "4"))    # erwartet: Berechnung nicht möglich: could not convert string to float: 'zehn', danach None

### Exceptions auslösen

#### Aufgabe 7: Ungültige Temperatur zurückweisen

Schreibe eine Funktion `celsius_to_kelvin(celsius)`, die eine Temperatur von Grad Celsius in Kelvin umrechnet (`kelvin = celsius + 273.15`). Liegt die Temperatur unter dem absoluten Nullpunkt (`-273.15` °C), soll die Funktion einen `ValueError` mit einer verständlichen Nachricht auslösen.

Rufe die Funktion anschließend mit `-300` in einem `try`/`except`-Block auf und gib die Nachricht aus.

In [ ]:
def celsius_to_kelvin(celsius):
    """Return celsius converted to kelvin."""
    # TODO: ungültige Werte mit raise zurückweisen
    # TODO: Umrechnung zurückgeben
    pass


print(celsius_to_kelvin(0))      # erwartet: 273.15
print(celsius_to_kelvin(-273.15)) # erwartet: 0.0

# TODO: celsius_to_kelvin(-300) aufrufen und den ValueError abfangen
# erwartet z.B.: Ungültige Eingabe: -300 °C liegt unter dem absoluten Nullpunkt.

#### Aufgabe 8: Abhebungen prüfen

Schreibe eine Funktion `withdraw(balance, amount)`, die den neuen Kontostand zurückgibt. Die Funktion soll einen `ValueError` auslösen, wenn

- `amount` nicht größer als `0` ist oder
- `amount` größer als `balance` ist.

Verarbeite danach alle Beträge aus `requested_amounts` nacheinander. Erfolgreiche Abhebungen verringern `balance`, abgelehnte Abhebungen werden mit ihrer Nachricht ausgegeben.

Erwartete Ausgabe (Nachrichten sinngemäß):
```
30 € abgehoben, neuer Kontostand: 70 €
Abgelehnt: Der Betrag muss größer als 0 sein.
Abgelehnt: Nicht genug Guthaben für 200 €.
50 € abgehoben, neuer Kontostand: 20 €
```

In [ ]:
def withdraw(balance, amount):
    """Return the new balance after withdrawing amount."""
    # TODO: beide ungültigen Fälle mit raise ValueError(...) zurückweisen
    # TODO: neuen Kontostand zurückgeben
    pass


balance = 100
requested_amounts = [30, -5, 200, 50]

for amount in requested_amounts:
    # TODO: withdraw() in try aufrufen, balance aktualisieren bzw. Fehlermeldung ausgeben
    pass

#### Aufgabe 9: Fehler melden und weiterreichen

Schreibe eine Funktion `get_setting(settings, key)`, die den Wert zu `key` aus dem Dictionary `settings` zurückgibt. Fehlt der Schlüssel, soll die Funktion `Einstellung '<key>' fehlt.` ausgeben und den `KeyError` anschließend **erneut auslösen**.

Die aufrufende Stelle fängt den `KeyError` ab und verwendet stattdessen den Standardwert `"de"`.

In [ ]:
settings = {"theme": "dark", "font_size": 14}


def get_setting(settings, key):
    """Return the value for key and re-raise a KeyError if key is missing."""
    # TODO: Zugriff in try, im except-Block Meldung ausgeben und erneut auslösen
    pass


print(get_setting(settings, "theme"))   # erwartet: dark

# TODO: get_setting(settings, "language") aufrufen, KeyError abfangen und "de" verwenden
language = None

print(language)   # erwartet: Einstellung 'language' fehlt., danach de

### Faustregeln

#### Aufgabe 10: Verschluckte Exception finden

Die Funktion `calculate_total()` liefert ein falsches Ergebnis, ohne dass eine Fehlermeldung erscheint. Führe die Zelle aus und überlege, warum. Entferne dann den `try`/`except`-Block, lies den Traceback und behebe die eigentliche Ursache.

In [ ]:
def calculate_total(prices):
    """Return the sum of all prices."""
    total = 0
    try:
        for price in prices:
            total = totl + price
    except:
        pass
    return total


print(calculate_total([4.5, 2.5, 7.0]))   # erwartet: 14.0

#### Aufgabe 11: Abfangen, crashen lassen oder selbst raisen?

Entscheide für jede Situation, welche der drei Faustregeln aus Kapitel 5 passt, und begründe die Entscheidung kurz im Kommentar.

1. Ein Programm liest beim Start die optionale Datei `settings.txt`. Fehlt sie, sollen Standardeinstellungen verwendet werden.
2. Eine eigene Funktion `calculate_bmi(weight, height)` wird mit `height=0` aufgerufen.
3. Beim Testen tritt ein `TypeError` auf, weil eine eigene Funktion versehentlich mit einer Liste statt mit einer Zahl aufgerufen wird.
4. Ein Programm fragt das Alter über `input()` ab und bekommt `"abc"` statt einer Zahl.
5. Eine Schleife greift wegen eines Denkfehlers bei der Indexberechnung auf einen Index hinter dem Listenende zu.

In [ ]:
# TODO: Entscheidung (abfangen / crashen lassen / selbst raisen) und Begründung ergänzen
# 1.
# 2.
# 3.
# 4.
# 5.

### Optional: Vertiefung

#### Aufgabe 12: `else` und `finally` kombinieren

Schreibe eine Funktion `process_payment(amount_text)`, die `amount_text` mit `float()` umwandelt. Ergänze den `try`-Block um folgende Blöcke:

- `except ValueError`: `Ungültiger Betrag: <amount_text>` ausgeben
- `else`: `Zahlung über <Betrag> € verbucht.` ausgeben
- `finally`: `Vorgang abgeschlossen.` ausgeben

Erwartete Ausgabe:
```
Zahlung über 19.99 € verbucht.
Vorgang abgeschlossen.
Ungültiger Betrag: neunzehn
Vorgang abgeschlossen.
```

In [ ]:
def process_payment(amount_text):
    """Convert amount_text and report whether the payment was booked."""
    # TODO: try / except / else / finally
    pass


process_payment("19.99")
process_payment("neunzehn")

#### Aufgabe 13: Eigene Exception für den Lagerbestand

Definiere eine eigene Exception-Klasse `OutOfStockError`. Schreibe anschließend eine Funktion `order_product(stock, product, quantity)`, die den Bestand im Dictionary `stock` um `quantity` verringert. Ist nicht genug auf Lager, soll die Funktion eine `OutOfStockError` mit einer verständlichen Nachricht auslösen.

Fange die Exception beim zweiten Aufruf ab und gib die Nachricht aus.

In [ ]:
# TODO: Klasse OutOfStockError definieren (erbt von Exception)


def order_product(stock, product, quantity):
    """Reduce the stock of product by quantity."""
    # TODO: bei zu geringem Bestand OutOfStockError auslösen
    # TODO: Bestand verringern
    pass


stock = {"Kaffee": 5, "Brot": 2}

order_product(stock, "Kaffee", 3)
print(stock)   # erwartet: {'Kaffee': 2, 'Brot': 2}

# TODO: order_product(stock, "Brot", 4) aufrufen und OutOfStockError abfangen
# erwartet z.B.: Bestellung abgelehnt: Nur noch 2 x Brot auf Lager, 4 angefordert.